# The human factors then become modifiers of that core finding:
Narrative Thread -  individually optimal behaviour is collectively dangerous in an emergency, and human factors amplify that danger.


* Walking speed — slower agents are more exposed because they clear the danger zone later. Does GA route slow agents differently?
* Delayed starts — some agents start evacuating later (panic, confusion). Does greedy's congestion get worse when agents arrive in waves?
* Anxiety — could model as agents ignoring their assigned GA path and defaulting to greedy. This is essentially your compliance question from SQ3.
* Familarity - agents only know a subset of exits (locals vs visitors). Does greedy's performance degrade faster than GA as exit knowledge decreases?

# Human Behaviour Characteristics

| Characteristic | Research Question | Human Framing | Implementation | Expected Result | Key Metrics to Compare | Risk |
|---|---|---|---|---|---|---|
| **Walking Speed** | Does GA route slow agents differently to fast agents, and does this affect collective outcomes? | Physical capability — elderly, injured, children vs able bodied adults. Slower agents remain in the danger zone longer. | Add speed parameter to agents. Each edge traversal costs `speed` timesteps in `get_timesteps`. Test speeds [1, 2, 3]. | Greedy degrades as slow agents congest exits. GA may naturally route slow agents to closer exits and fast agents to further ones. | Congestion index, avg delay by speed group, start node dispersal by speed group, total evacuation time | GA may not learn speed-aware routing without explicit speed consideration in fitness function |
| **Delayed Starts** | Does greedy's congestion worsen when agents start evacuating in waves vs simultaneously? | Panic, disbelief, waiting for confirmation — not all agents react immediately to an emergency. | Add `start_delay` parameter to agents drawn from a distribution. Agents do not enter simulation until their delay timestep. Test delay distributions [none, low variance, high variance]. | Greedy degrades significantly — late starters walk into congestion created by early starters with no ability to reroute. GA paths pre-planned so late starters follow assigned route. | Total evacuation time, congestion index over time, cumulative exits plot, start node dispersal | Interaction with walking speed may complicate results |
| **Compliance/Anxiety** | At what compliance rate does collective GA routing begin to outperform greedy? | Anxious agents abandon pre-planned routes and default to nearest exit regardless of instructions. | At initialisation each agent rolls against compliance rate. Non-compliant agents get greedy shortest path and are frozen — GA cannot modify their paths. All agents included in congestion calculation. Test rates [0.0, 0.25, 0.5, 0.75, 1.0]. | Clear threshold compliance rate where GA begins to outperform greedy. Below threshold non-compliant agents create unexpected congestion at exits GA assumed underutilised. | Total evacuation time, congestion index, exit utilisation, cumulative exits plot at each compliance rate | With small population size threshold may not be clean — run multiple seeds |
| **Familiarity** | Does greedy degrade faster than GA as agents' exit knowledge decreases? | Locals vs visitors — locals know all exits, tourists only know the entrance they arrived through. Unfamiliar occupants have significantly worse evacuation outcomes (well documented in literature). | Restrict `preferred_exits` at agent initialisation. Test familiarity levels [all exits, 2 exits, 1 exit]. Greedy routes to nearest known exit only. GA optimises across known exits only. | Greedy degrades faster than GA as familiarity decreases — at 1 exit everyone funnels to same exit regardless of congestion. GA's advantage over greedy grows as familiarity decreases. | Total evacuation time, exit utilisation, congestion index, cumulative exits plot at each familiarity level | With only 4 exits in moderate city the difference between 2 and 4 exits may be small |


#### Research 
* https://pmc.ncbi.nlm.nih.gov/articles/PMC10935319/
* https://link.springer.com/article/10.1186/2193-0414-2-7 = *"Future research and model developments should focus on the study of the impact of staff actions, group dynamics and people with disabilities."*
* https://www.sciencedirect.com/science/article/abs/pii/S0951832014001628 = Both experimental and simulation data on fire evacuation are influenced by a component of uncertainty caused by the impact of the unexplained variance in human behaviour, namely behavioural uncertainty (BU).
* https://www.cell.com/heliyon/fulltext/S2405-8440(23)01482-2 = vr to refine outputs of agent evacuation models.

### Walking Speed
```
Ran the different combinations of walking speed and congestion on and off and get the following outputs for the grid city.
```
The combined effect being slightly larger than the sum of individual effects is expected — slow agents occupying nodes longer creates more congestion opportunities.
What's interesting:

Exit utilisation drops when both are added (0.32 → 0.20) — the GA is struggling to optimise as effectively under combined constraints, which makes sense with epsilon=0.2
Congestion index jumps from 60% to 75% when walking speed is added — confirms your earlier point that slow agents blocking nodes inflates congestion
Path efficiency stays close to 1.0 with just walking, meaning the GA isn't taking unnecessarily long paths to avoid walking delay — it correctly identifies that walking speed is unavoidable


#### Research
The current implementation uses the following split:
```          
options=[1, 2, 3],
weights=[0.6, 0.3, 0.1]
```
* Fruin, J.J (1971) -  Pedestrian Planning and Design --> established a mean walking speed of 1.35 m/s for pedestrians Thunderheadeng, which is the standard baseline cited across virtually all evacuation research. = https://onlinepubs.trb.org/Onlinepubs/hrr/1971/355/355-001.pdf
* Peacock R.D, Hoskins B.L & Kuligowski E.D (NIST) -  Overall and Local Movement Speeds During Fire Drill Evacuations --> The study found that 19% of occupants moved slower than 0.4 m/s, with the majority of speeds falling between 0.3 and 0.7 m/s during fire drill evacuations.
* Probably should look into this papers introduction section - https://link.springer.com/rwe/10.1007/978-0-387-30440-3_382
* https://journals.sagepub.com/doi/abs/10.1177/154193120605001118 - Muhdi et al. measured the average walking speed, reporting an average of 1.32 m/s under non-emergency conditions, but found it could increase to a maximum of 2.16 m/s during emergencies.
* https://www.sciencedirect.com/science/article/abs/pii/S0375960117307399 = Zhao et al. also recorded average walking speeds of 1.32 m/s under normal conditions, as well as a maximum of 2.91 m/s during emergencies.

This experiement assume a walking speed as a scale i.e 1 is free walking speed rather than a time per unit measurement. It also ignores additional conditions such as the requirement to crawl to avoid harmful by products of evacuations, visibility or ground conditions, and assumes that any free walking speed delay would also be the same if the agents needs to crawl.
* Crawling - https://www.sciencedirect.com/science/article/abs/pii/S0379711208001331, https://www.researchgate.net/profile/Jerry-Davis-5/publication/268802219_The_Impact_of_Posture_on_Evacuation_Speed/links/54a2e64e0cf257a63604db24/The-Impact-of-Posture-on-Evacuation-Speed.pdf
* Visibility - https://www.sciencedirect.com/science/article/abs/pii/S0360132310003410, https://www.mdpi.com/2412-3811/7/4/57


### Delayed Starts
....


#### Research
The current implementation uses the following split:
```          
options=[0, 1, 2],
weights=[0.4, 0.4, 0.2],
```

* Ronchi et al. (2019) — The Variation of Pre-movement Time in Building Evacuation, Fire Technology --> Analysing 2486 data points across 40 unannounced evacuation experiments, the study found pre-movement times follow a lognormal or loglogistic distribution with a rapid initial increase followed by a slower tail. 
* https://www.researchgate.net/profile/Rita-Fahy/publication/44082840_Toward_creating_a_database_on_delay_times_to_start_evacuation_and_walking_speeds_for_use_in_evacuation_modeling/links/5968c687aca2728ca67be538/Toward-creating-a-database-on-delay-times-to-start-evacuation-and-walking-speeds-for-use-in-evacuation-modeling.pdf = Studies have shown that the time occupants will delay can vary according to the cue they receive (alarm bells, warnings by staff, voice announcements or smoke, for example).
* Analysing 2486 data points across 40 unannounced evacuation experiments, the study found pre-movement times follow a lognormal or loglogistic distribution with a rapid initial increase followed by a slower tail. 
* https://link.springer.com/chapter/10.1007/978-1-4939-2565-0_58 = timeline to human response (good diagram).
* Overall, Proulx found that 62 % of the occupants (in the four buildings studied) evacuated in groups. = https://www.sciencedirect.com/science/article/abs/pii/037971129500023M

Ronchi et al - lognormal justifies the simplified right skewed delay times.
weights as a sensitivity parameter rather than a demographic claim — something like "weights were varied to reflect populations with differing proportions of mobility-impaired occupants." That's actually stronger methodologically because it means you can test [0.8, 0.15, 0.05] vs [0.6, 0.3, 0.1] and show how the GA/greedy gap changes with population composition. 